# Notebook 2: Latency Benchmark

**Purpose:** Benchmark all 4 VLM candidates on Jetson Thor. Measure p50/p95/p99 latency
and TTFT (time to first token) separately. Compare free-form vs. guided JSON output.
Eliminate models that can't hit p95 < 800ms.

**Prerequisites:** Notebook 1 occlusion gate must PASS before running this.

**Decision rules:**
- p95 < 800ms → passes, moves to Notebook 3
- p95 800–999ms → marginal, moves to Notebook 3 but flagged as latency risk
- p95 >= 1000ms → eliminated
- error_rate > 5% → flagged as unreliable regardless of latency

**Day 2 budget:** ~3–4 hours for 4 models × 2 prompt strategies + swap overhead.
Expand ground truth dataset from 20 to 30 images while benchmark runs.

In [ ]:
# ---- CELL 1: Environment check ----
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', '../requirements.txt'])

import pkg_resources
print('=== Environment ===')
for pkg in ['openai', 'Pillow', 'scikit-learn', 'numpy', 'tqdm']:
    try:
        print(f'  {pkg}: {pkg_resources.get_distribution(pkg).version}')
    except Exception:
        print(f'  {pkg}: NOT FOUND')
try:
    import vllm; print(f'  vllm: {vllm.__version__}')
except ImportError:
    pass
print('Done.')

In [ ]:
# ---- CELL 2: Configuration ----
# Update model IDs after verifying on HuggingFace (Day 0 checklist item).

from pathlib import Path

# Models to benchmark — verify all IDs on HuggingFace before Day 0 download
CANDIDATE_MODELS = {
    'qwen25vl_7b_awq':   'Qwen/Qwen2.5-VL-7B-Instruct-AWQ',
    'cosmos_nemotron':   'nvidia/Cosmos-Nemotron-34B-Vision',  # TODO: verify ID at hf.co/nvidia
    'vila_7b':           'Efficient-Large-Model/VILA1.5-7b',   # may need --trust-remote-code
    'llava_13b':         'llava-hf/llava-1.5-13b-hf',          # FP16; confirm GPTQ availability
}

# vLLM settings
VLLM_HOST = 'localhost'
VLLM_PORT = 8000
VLLM_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}/v1'

# Benchmark parameters
N_WARMUP = 3     # discarded warm-up runs before measurement window
N_RUNS   = 50    # measurement window
MAX_ERROR_RATE = 0.05  # 5% — flag model as unreliable if exceeded

# Latency thresholds (ms)
PASS_THRESHOLD     = 800   # p95 < this → passes
MARGINAL_THRESHOLD = 1000  # p95 < this → marginal (flagged)
# p95 >= MARGINAL_THRESHOLD → eliminated

# Model swap timeout
SWAP_MEMORY_TIMEOUT_S = 1200  # 20 minutes
VLLM_STARTUP_TIMEOUT_S = 300  # 5 minutes

IMAGES_DIR = Path('../data/images')

print(f'Models to benchmark: {list(CANDIDATE_MODELS.keys())}')
print(f'Warmup runs: {N_WARMUP}, Measurement runs: {N_RUNS}')
print(f'Pass threshold: p95 < {PASS_THRESHOLD}ms')

In [ ]:
# ---- CELL 3: Helper functions ----

import base64, io, time, json, requests, signal
import numpy as np
from PIL import Image
from openai import OpenAI
from typing import Optional

def image_to_b64(image_path: Path, max_size: int = 1024) -> str:
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    if max(w, h) > max_size:
        scale = max_size / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=90)
    return base64.b64encode(buf.getvalue()).decode('utf-8')

def make_vision_message(b64_image: str, prompt: str) -> list:
    return [{'role': 'user', 'content': [
        {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64_image}'}},
        {'type': 'text', 'text': prompt},
    ]}]

def wait_for_vllm(url: str, timeout_s: int = 300, poll_s: int = 5) -> bool:
    health_url = url.replace('/v1', '/health')
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            if requests.get(health_url, timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        print(f'  waiting for vLLM... ({int(deadline - time.time())}s left)', end='\r')
        time.sleep(poll_s)
    return False

def wait_for_gpu_clear(timeout_s: int = SWAP_MEMORY_TIMEOUT_S, threshold_mb: int = 2048) -> bool:
    """Poll until GPU memory drops below threshold. Returns True if cleared."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                text=True
            ).strip()
            used_mb = int(out.split('\n')[0])
            print(f'  GPU memory: {used_mb} MB', end='\r')
            if used_mb < threshold_mb:
                return True
        except Exception:
            return True  # nvidia-smi unavailable or unified memory — assume cleared
        time.sleep(5)
    return False

print('Helpers loaded.')

In [ ]:
# ---- CELL 4: Board state schema (array format, used for guided_json) ----
#
# Array format chosen over variable-key map for faster guided_json generation.
# Uses JSON Schema `items` path (fast) instead of `additionalProperties` path (slow).
#
# Only OCCUPIED squares are listed. Absence from array = unoccupied.

PIECE_TYPE_ENUM = ['house', 'hotel', 'token_red', 'token_blue', 'token_green', 'token_yellow']
# Add other token colors your physical set uses:
# PIECE_TYPE_ENUM += ['token_purple', 'token_white']

BOARD_STATE_SCHEMA = {
    'type': 'object',
    'properties': {
        'pieces': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'square_id':  {'type': 'string', 'pattern': '^[0-3][0-9]$'},
                    'piece_type': {'type': 'string', 'enum': PIECE_TYPE_ENUM},
                    'owner':      {'type': 'string'},
                },
                'required': ['square_id', 'piece_type', 'owner'],
                'additionalProperties': False,
            },
        }
    },
    'required': ['pieces'],
    'additionalProperties': False,
}

FREEFORM_PROMPT = (
    'Look at this Monopoly board image. '
    'Describe which squares have pieces or buildings on them.'
)

GUIDED_PROMPT = (
    'Look at this Monopoly board image. '
    'Return a JSON object listing every occupied square. '
    'Square IDs are zero-padded numbers: 00=Go, 39=Boardwalk. '
    'Only include squares that have a piece or building on them.'
)

print('Schema defined.')
print(f'Piece types: {PIECE_TYPE_ENUM}')

In [ ]:
# ---- CELL 5: guided_json activation check ----
# Sends 1 test request with guided_json before the 50-run loop.
# If the response isn't valid JSON matching the schema, the grammar backend failed.
# Aborts this model's benchmark if grammar is not active — prevents silent wrong data.

def check_guided_json_active(client: OpenAI, model_id: str, b64_image: str) -> bool:
    """Returns True if guided_json is working. False if backend fell back to free-form."""
    try:
        resp = client.chat.completions.create(
            model=model_id,
            messages=make_vision_message(b64_image, GUIDED_PROMPT),
            max_tokens=512,
            temperature=0.0,
            extra_body={'guided_json': BOARD_STATE_SCHEMA},
        )
        text = resp.choices[0].message.content
        parsed = json.loads(text)
        # Must have 'pieces' key and it must be a list
        return isinstance(parsed.get('pieces'), list)
    except (json.JSONDecodeError, Exception) as e:
        print(f'  guided_json check failed: {e}')
        return False

print('guided_json activation check function ready.')
print('Will be called before each model\'s 50-run loop.')

In [ ]:
# ---- CELL 6: Benchmark function ----
# Runs N_WARMUP discarded runs, then N_RUNS measurement runs.
# Measures total_latency and TTFT (time to first token) separately.
# Errors are excluded from latency histograms and tracked as error_rate.

def run_benchmark(client: OpenAI, model_id: str, b64_image: str,
                  use_guided_json: bool) -> dict:
    """
    Returns:
      latencies_ms: list of total latency per successful run
      ttft_ms:      list of time-to-first-token per successful run (streaming)
      error_count:  number of failed requests
      error_rate:   fraction of failed requests
    """
    prompt = GUIDED_PROMPT if use_guided_json else FREEFORM_PROMPT
    extra = {'guided_json': BOARD_STATE_SCHEMA} if use_guided_json else {}
    messages = make_vision_message(b64_image, prompt)

    latencies_ms = []
    ttft_ms = []
    error_count = 0
    total_attempts = N_WARMUP + N_RUNS

    for i in range(total_attempts):
        is_warmup = i < N_WARMUP
        label = f'warmup {i+1}/{N_WARMUP}' if is_warmup else f'run {i-N_WARMUP+1}/{N_RUNS}'

        try:
            t_start = time.perf_counter()
            first_token_time = None

            # Use streaming to capture TTFT
            stream = client.chat.completions.create(
                model=model_id,
                messages=messages,
                max_tokens=512,
                temperature=0.0,
                stream=True,
                extra_body=extra,
            )
            for chunk in stream:
                if first_token_time is None and chunk.choices and chunk.choices[0].delta.content:
                    first_token_time = time.perf_counter()
            t_end = time.perf_counter()

            total_ms = (t_end - t_start) * 1000
            ttft = (first_token_time - t_start) * 1000 if first_token_time else None

            if not is_warmup:
                latencies_ms.append(total_ms)
                if ttft is not None:
                    ttft_ms.append(ttft)
            print(f'  [{label}] total={total_ms:.0f}ms ttft={ttft:.0f}ms' if ttft else
                  f'  [{label}] total={total_ms:.0f}ms', end='\r')

        except Exception as e:
            if not is_warmup:
                error_count += 1
            print(f'  [{label}] ERROR: {e}')

    error_rate = error_count / N_RUNS if N_RUNS > 0 else 0.0
    return {
        'latencies_ms': latencies_ms,
        'ttft_ms': ttft_ms,
        'error_count': error_count,
        'error_rate': error_rate,
    }

print('Benchmark function ready.')

In [ ]:
# ---- CELL 7: Model swap function ----
# Kills the current vLLM process, waits for GPU memory to clear,
# then starts vLLM with the next model.

def build_vllm_cmd(model_id: str) -> list:
    cmd = [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', model_id,
        '--skip-mm-profiling',              # SM100 workaround (issue #38411)
        '--attention-backend', 'TRITON_ATTN',  # SM100 workaround
        '--dtype', 'auto',
        '--host', VLLM_HOST,
        '--port', str(VLLM_PORT),
    ]
    # Some models need --trust-remote-code (e.g., VILA)
    if 'vila' in model_id.lower():
        cmd.append('--trust-remote-code')
    return cmd

current_proc = None  # global handle

def swap_model(model_id: str) -> Optional[subprocess.Popen]:
    """Stop current vLLM, wait for GPU clear, start new model. Returns process or None on failure."""
    global current_proc

    # 1. Terminate existing process
    if current_proc is not None and current_proc.poll() is None:
        print(f'Stopping vLLM...')
        current_proc.terminate()
        try:
            current_proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            current_proc.kill()
        current_proc = None

    # 2. Wait for GPU memory to free
    print('Waiting for GPU memory to clear...')
    cleared = wait_for_gpu_clear(timeout_s=SWAP_MEMORY_TIMEOUT_S)
    if not cleared:
        print(f'WARNING: GPU memory did not clear within {SWAP_MEMORY_TIMEOUT_S//60} minutes.')
        print('Attempting to start next model anyway — may OOM.')
    else:
        print('\nGPU memory cleared.')

    # 3. Start new vLLM process
    cmd = build_vllm_cmd(model_id)
    print(f'Starting vLLM with model: {model_id}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    ready = wait_for_vllm(VLLM_BASE_URL, timeout_s=VLLM_STARTUP_TIMEOUT_S)
    if not ready:
        proc.terminate()
        print(f'ERROR: vLLM did not start for model {model_id}. Skipping.')
        return None

    print(f'\nvLLM ready: {model_id}')
    current_proc = proc
    return proc

print('swap_model() ready.')

In [ ]:
# ---- CELL 8: Load benchmark image ----
# Use one representative mid-game board image for the latency benchmark.
# Same image for all models ensures fair comparison.

bench_images = sorted(IMAGES_DIR.glob('*.jpg')) + sorted(IMAGES_DIR.glob('*.jpeg'))
if not bench_images:
    raise FileNotFoundError(f'No images found in {IMAGES_DIR}. Complete Day 0 first.')

# Use the first image. Change index to pick a different one.
BENCH_IMAGE_PATH = bench_images[0]
BENCH_IMAGE_B64 = image_to_b64(BENCH_IMAGE_PATH)
print(f'Benchmark image: {BENCH_IMAGE_PATH.name}')

In [ ]:
# ---- CELL 9: Main benchmark loop ----
# Runs all candidate models sequentially.
# For each model:
#   1. Swap to the model
#   2. Check guided_json is active
#   3. Run free-form benchmark
#   4. Run guided_json benchmark
#   5. Record results

all_results = {}  # model_key -> {freeform, guided} -> benchmark data

for model_key, model_id in CANDIDATE_MODELS.items():
    print(f'\n{"="*60}')
    print(f'Model: {model_key} ({model_id})')
    print(f'{'='*60}')

    # Swap to this model
    proc = swap_model(model_id)
    if proc is None:
        print(f'SKIPPED: {model_key} failed to load.')
        all_results[model_key] = {'status': 'load_failed', 'model_id': model_id}
        continue

    client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-required')

    # guided_json activation check
    print('Checking guided_json activation...')
    guided_active = check_guided_json_active(client, model_id, BENCH_IMAGE_B64)
    if not guided_active:
        print(f'WARNING: guided_json not active for {model_key}.')
        print('Guided-JSON benchmark will be skipped for this model.')
        print('This means structured output will not work in production — review carefully.')
    else:
        print('guided_json: ACTIVE')

    model_results = {'status': 'ok', 'model_id': model_id, 'guided_json_active': guided_active}

    # Free-form benchmark
    print(f'\nRunning free-form benchmark ({N_WARMUP} warmup + {N_RUNS} runs)...')
    ff = run_benchmark(client, model_id, BENCH_IMAGE_B64, use_guided_json=False)
    model_results['freeform'] = ff

    # guided_json benchmark
    if guided_active:
        print(f'\nRunning guided_json benchmark ({N_WARMUP} warmup + {N_RUNS} runs)...')
        gj = run_benchmark(client, model_id, BENCH_IMAGE_B64, use_guided_json=True)
        model_results['guided'] = gj
    else:
        model_results['guided'] = {'status': 'skipped_no_grammar'}

    all_results[model_key] = model_results
    print(f'\nDone: {model_key}')

# Shutdown the last model
if current_proc is not None and current_proc.poll() is None:
    current_proc.terminate()
    current_proc.wait()
    print('\nAll models benchmarked. vLLM stopped.')

In [ ]:
# ---- CELL 10: Results table ----

import numpy as np

def summarize(data: list) -> dict:
    if not data:
        return {'p50': None, 'p95': None, 'p99': None}
    a = np.array(data)
    return {
        'p50': float(np.percentile(a, 50)),
        'p95': float(np.percentile(a, 95)),
        'p99': float(np.percentile(a, 99)),
    }

def latency_verdict(p95_ms: Optional[float], error_rate: float) -> str:
    if p95_ms is None:
        return 'N/A'
    if error_rate > MAX_ERROR_RATE:
        return f'UNRELIABLE (err={error_rate:.0%})'
    if p95_ms < PASS_THRESHOLD:
        return 'PASS'
    if p95_ms < MARGINAL_THRESHOLD:
        return 'MARGINAL'
    return 'ELIMINATED'

print(f'{'Model':<22} {'Strategy':<12} {'p50':>7} {'p95':>7} {'p99':>7} {'TTFT p95':>9} {'Err%':>6}  Verdict')
print('-' * 90)

passing_models = []

for model_key, res in all_results.items():
    if res.get('status') == 'load_failed':
        print(f'{model_key:<22} {"LOAD FAILED":<70}')
        continue

    for strategy, key in [('free-form', 'freeform'), ('guided_json', 'guided')]:
        d = res.get(key, {})
        if d.get('status') == 'skipped_no_grammar':
            print(f'{model_key:<22} {strategy:<12} {"(grammar inactive)":>50}')
            continue
        lats = d.get('latencies_ms', [])
        ttfts = d.get('ttft_ms', [])
        err_rate = d.get('error_rate', 0.0)
        s = summarize(lats)
        t = summarize(ttfts)
        verdict = latency_verdict(s['p95'], err_rate)
        p50  = f"{s['p50']:.0f}ms" if s['p50'] else 'N/A'
        p95  = f"{s['p95']:.0f}ms" if s['p95'] else 'N/A'
        p99  = f"{s['p99']:.0f}ms" if s['p99'] else 'N/A'
        ttft95 = f"{t['p95']:.0f}ms" if t['p95'] else 'N/A'
        print(f'{model_key:<22} {strategy:<12} {p50:>7} {p95:>7} {p99:>7} {ttft95:>9} {err_rate:>5.0%}  {verdict}')

        if key == 'guided' and verdict in ('PASS', 'MARGINAL') and res.get('guided_json_active'):
            passing_models.append(model_key)

print('-' * 90)
print(f'\nModels advancing to Notebook 3: {passing_models or "NONE — all eliminated"}')
if not passing_models:
    print('\nNO MODELS PASSED. Consider:')
    print('  - Switching to a smaller/more quantized model')
    print('  - Investigating if --skip-mm-profiling is affecting latency')
    print('  - Checking if TRT Edge-LLM would give better performance')

In [ ]:
# ---- CELL 11: Save results ----
import json
from datetime import datetime

# Serialize results (convert numpy types)
def serialize(obj):
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, list):
        return [serialize(x) for x in obj]
    if isinstance(obj, dict):
        return {k: serialize(v) for k, v in obj.items()}
    return obj

out_path = Path('../data') / f'latency_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(out_path, 'w') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'config': {
            'n_warmup': N_WARMUP,
            'n_runs': N_RUNS,
            'pass_threshold_ms': PASS_THRESHOLD,
            'marginal_threshold_ms': MARGINAL_THRESHOLD,
            'benchmark_image': str(BENCH_IMAGE_PATH.name),
        },
        'results': serialize(all_results),
        'passing_models': passing_models,
    }, f, indent=2)

print(f'Results saved to {out_path}')
print(f'Pass these model keys to Notebook 3: {passing_models}')